# 06 · CP3: training health, seed 42 (Milestone 3; spec section 9.1)

Runs on an **L4**. Run it **twice**: first with `PROFILE = 'unified'`, then with
`PROFILE = 'detection'` (cell 2). Seeds 123 and 2026 come only after CP3 and CP4 are accepted.

| Run | Steps per epoch (batch 32) | Epochs (most) | Time (most) | Compute units at 1.7/h |
|---|---|---|---|---|
| unified: A1 overfit, then training | 750 | 20 (early stop, patience 4) | ≈ 3 h | ≈ 5.3 |
| detection | 500 | 15 | ≈ 1.4 h | ≈ 2.3 |
| set-up per session (shard copy, CLIP download) | | | ≈ 8 min | ≈ 0.2 |

**Check your compute-unit balance in Colab's resource panel before starting (about 8 units for both runs).**

What it does:

1. set-up; 2. settings (the CLIP commit and hash come from `configs/base.yaml`, no override);
3. copies the shards to the local disk;
4. **A1** (unified only): the 64-image overfit with fixed crops, augmentation and dropout off —
   accuracy ≥ 99% within 300 steps and per-image mask F1 on the 22 edited images ≥ 0.80 within
   1,000 steps. Its record is reused when it exists. **If A1 fails, training does not start**:
   send `overfit_a1.json` to Claude;
5. training **in the foreground** with a progress bar per epoch; a checkpoint every 500 steps and at
   each epoch end. **If the session drops, run cells 1–5 again: training resumes** from `last.pt`;
6. curves (loss, AUC, mask F1, marks on real and generated crops), saved as `curves.png`;
7. the CP3 evaluation of every criterion (S1–S3, A1–A4, R1, R2) at the best epoch.

**Results** (send to Claude):
- `Paper 1/runs/checkpoints/cp3_training_health_<profile>_seed42.json`;
- the run folder `Paper 1/runs/eviground_<profile>_seed42/` (`metrics.csv`, `steps.csv`,
  `compute.json`, `curves.png`), which Claude can read on Drive while a run is going.

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings (run once with PROFILE = 'unified', then once with 'detection')
import datetime, json, logging, os, time
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
import torch
from eviground.config import load_config
from eviground.data import pack
from eviground.env import get_paths, session_info

PROFILE = os.environ.get('EVIGROUND_PROFILE', 'unified')   # 'unified' first, then 'detection'
SEED = 42                  # seeds 123 and 2026 only after CP3 and CP4 are accepted
CU_PER_HOUR = 1.7          # read the live rate in Colab's resource panel and update
assert PROFILE in ('unified', 'detection'), PROFILE
T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
if DRY:   # fake world, fake shards, tiny CLIP, CPU
    import shutil, tempfile
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / f'eviground_cp3_dry_{PROFILE}'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
paths.ensure()
CONFIG = 'debug' if DRY else 'base'
# Hugging Face's cache uses symbolic links, which Google Drive cannot store: local disk
HF_CACHE = paths.cache / 'hf' if DRY else Path('/content/hf_cache')
OVERRIDES = [f'model.semantic.cache_dir={HF_CACHE}', f'compute.cu_per_hour={CU_PER_HOUR}']
cfg = load_config(CONFIG, OVERRIDES)
if not DRY:
    assert torch.cuda.is_available(), 'CP3 needs a GPU runtime (L4)'
    assert cfg.model.semantic.revision and cfg.model.semantic.weights_sha256, \
        'pin the CLIP commit and weights hash in configs/base.yaml'
assert not cfg.model.trace.get('channels_last', False), 'channels_last stays off (15 Nov)'
REVISION = cfg.model.semantic.revision if not DRY else 'tiny'
EXPERIMENT = cfg.get('experiment', 'eviground')
RUN = paths.runs / f'{EXPERIMENT}_{PROFILE}_seed{SEED}'
print('profile', PROFILE, '| seed', SEED, '| commit', info['git_commit'], '| gpu', info.get('gpu_name'),
      '| CLIP revision', REVISION, '| run folder', RUN)

In [ ]:
# Cell 3: the shards (dry run: fake world -> fake CP1 run -> pack), copied to the local disk
SHARDS = paths.shards
if DRY:
    from eviground.data.cp1 import CP1Config, run_cp1
    from eviground.data.download import ensure_data
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', coco_images=40)
    ensure_data('cp1', paths=paths, sources=sources, stage_local=True, overrides=cp1_overrides(40))
    out = paths.runs / 'checkpoints' / 'cp1'
    run_cp1(paths, CP1Config(workers=2, crops_per_class=60, blockiness_per_group=20,
                             base_unclear_max=0.2, diffseg_card_splits={'train': 18, 'validation': 12}),
            out_dir=out, cache_dir=paths.cache / 'cp1', progress=None)
    pack.pack(out / 'manifest_cp1.csv.gz', SHARDS, free_bytes=10**12, margin=0,
              shard_bytes=200_000, sample_per_source=5, progress=None)
done = pack.read_done(SHARDS)
assert done, f'no finished shards in {SHARDS}: run notebook 04 first'
print('shards:', done['images'], 'images |', done['mask_rule'])
# each read through Colab's Drive mount costs ~0.25 s: train from a local copy (checked shard by shard)
LOCAL_SHARDS = paths.local_data / 'shards'          # /content/data/shards on Colab
staged = pack.stage_shards(SHARDS, LOCAL_SHARDS, progress=print)
SHARDS = LOCAL_SHARDS
INDEX_SHA256 = pack.file_sha256(SHARDS / 'index.parquet')
print('staged:', staged)

In [ ]:
# Cell 4: A1, the 64-image overfit (unified only; reused when its record exists)
from eviground.train.trainer import NonFiniteLoss, Trainer

OVERFIT = None
if PROFILE == 'unified':
    of_dir = paths.runs / f'{EXPERIMENT}_unified_seed{SEED}_overfit'
    of_file = of_dir / 'overfit_a1.json'
    old = json.loads(of_file.read_text()) if of_file.is_file() else None
    if old and old.get('index_sha256') == INDEX_SHA256 and 'mask_pass' in old:
        OVERFIT = old
        print('A1: reusing', of_file)
    else:
        t = time.time()
        ot = Trainer(cfg, SHARDS, of_dir, seed=SEED, profile='unified', resume=None)
        OVERFIT = {**ot.overfit(), 'minutes': round((time.time() - t) / 60, 2),
                   'index_sha256': INDEX_SHA256, 'git_commit': info['git_commit'],
                   'clip_revision': REVISION, 'clip_sha256': ot.clip_sha256}
        of_dir.mkdir(parents=True, exist_ok=True)
        of_file.write_text(json.dumps(OVERFIT, indent=1, default=str))
        del ot
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print({k: v for k, v in OVERFIT.items() if k not in ('history', 'by_class')})
    for h in OVERFIT['history']:
        print(f"  step {h['step']:>5}  loss {h['loss']:.4f}  acc {h['acc']:.3f}  mask F1 {h['mask_f1']}")
    a1_ok = OVERFIT['acc_pass'] and OVERFIT['mask_pass']
    if not a1_ok and not DRY:
        raise RuntimeError(f'A1 failed: training not started. Send {of_file} to Claude.')
else:
    print('A1 is checked on the unified model only')

In [ ]:
# Cell 5: training in the foreground (progress bar per epoch). Re-running cells 1-5 resumes.
t = time.time()
tr = Trainer(cfg, SHARDS, RUN, seed=SEED, profile=PROFILE, resume='auto', progress=True)
print({'steps_per_epoch': tr.steps_per_epoch, 'epochs': tr.epochs, 'train_items': len(tr.train_ds),
       'val_items': len(tr.val_ds), 'train_eval_items': len(tr.train_eval_ds or []),
       'starting_at_global_step': tr.state['global_step'], 'epoch': tr.state['epoch']})
NONFINITE = None
try:
    SUMMARY = tr.fit()
except NonFiniteLoss as e:      # S1: no step was taken; last.pt and nonfinite.json are saved
    NONFINITE = str(e)
    SUMMARY = tr.summary()
    print('STOPPED:', e)
SUMMARY['session_training_minutes'] = round((time.time() - t) / 60, 2)
(RUN / 'summary.json').write_text(json.dumps(SUMMARY, indent=1, default=str))
print(json.dumps(SUMMARY, indent=1, default=str))
del tr
torch.cuda.empty_cache() if torch.cuda.is_available() else None

In [ ]:
# Cell 6: curves (saved as curves.png in the run folder)
from eviground.eval import cp3

rows = cp3.read_csv(RUN / 'metrics.csv')
steps = cp3.read_csv(RUN / 'steps.csv')
num = lambda r, k: float(r[k]) if r.get(k) not in (None, '') else float('nan')
try:
    import matplotlib.pyplot as plt
except ImportError:          # not installed in the CI dry run; Colab has it
    plt = None
    print('matplotlib not installed: no curves')
if plt is not None and rows:
    ep = [int(r['epoch']) + 1 for r in rows]
    fig, ax = plt.subplots(2, 2, figsize=(13, 8))
    a = ax[0, 0]
    a.plot([int(s['global_step']) for s in steps], [num(s, 'loss_total') for s in steps],
           lw=0.4, alpha=0.5, label='training loss per step')
    a.plot([int(r['global_step']) for r in rows], [num(r, 'train_total') for r in rows],
           'o-', label='training loss, epoch mean')
    a.set_xlabel('optimizer step'); a.set_title('S1 loss'); a.legend()
    a = ax[0, 1]
    auc = 'val_macro_auc' if PROFILE == 'unified' else 'val_det_auc'
    a.plot(ep, [num(r, 'val_selection') for r in rows], 'o-', label='validation selection')
    a.plot(ep, [num(r, 'train_eval_selection') for r in rows], 's--', label='training subset selection')
    a.plot(ep, [num(r, auc) for r in rows], '^-', label=auc)
    if PROFILE == 'unified':
        for c in ('real', 'generated', 'edited'):
            a.plot(ep, [num(r, f'val_auc_{c}') for r in rows], lw=1, label=f'AUC {c}')
    else:
        a.plot(ep, [num(r, 'val_det_acc') for r in rows], 'v-', label='SD v1.4 accuracy (S3)')
        a.axhline(0.95, ls=':', c='k')
    a.axhline(0.90, ls=':', c='grey'); a.set_xlabel('epoch'); a.set_title('S2 validation'); a.legend(fontsize=7)
    if PROFILE == 'unified':
        a = ax[1, 0]
        for k, lab in (('val_mask_f1', 'validation mask F1 (A4)'), ('train_eval_mask_f1', 'training subset mask F1'),
                       ('val_mask_miou_binary', 'validation binary mIoU (R2)')):
            a.plot(ep, [num(r, k) for r in rows], 'o-', label=lab)
        a.axhline(0.50, ls=':', c='k'); a.axhline(0.2162, ls=':', c='r', label='untrained (CP2)')
        a.set_xlabel('epoch'); a.set_title('A4 / R2 localization'); a.legend(fontsize=7)
        a = ax[1, 1]
        for k in ('val_real_any_pred', 'val_gen_any_pred', 'val_real_pred_area', 'val_gen_pred_area'):
            a.plot(ep, [num(r, k) for r in rows], 'o-', label=k[4:])
        a.axhline(0.20, ls=':', c='k'); a.axhline(0.01, ls=':', c='grey')
        a.set_yscale('symlog', linthresh=0.01); a.set_xlabel('epoch'); a.set_title('A2 / A3 marks on unedited crops')
        a.legend(fontsize=7)
    else:
        ax[1, 0].axis('off'); ax[1, 1].axis('off')
    fig.suptitle(f'{RUN.name}')
    fig.tight_layout()
    fig.savefig(RUN / 'curves.png', dpi=110)
    plt.show()
print(len(rows), 'epochs logged')

In [ ]:
# Cell 7: the CP3 evaluation (Dhanush decides the checkpoint) and the result file
REPORT = cp3.evaluate_run(RUN, PROFILE, OVERFIT if PROFILE == 'unified' else None)
compute_file = RUN / 'compute.json'
hours = (time.time() - T0) / 3600
REPORT.update({
    'step': 'CP3 training health', 'dry_run': DRY, 'branch': BRANCH, 'seed': SEED,
    'session': info, 'config': CONFIG, 'overrides': OVERRIDES, 'clip_revision': REVISION,
    'index_sha256': INDEX_SHA256, 'summary': SUMMARY, 'nonfinite': NONFINITE,
    'overfit_a1': {k: v for k, v in OVERFIT.items() if k != 'history'} if OVERFIT else None,
    'compute_run': json.loads(compute_file.read_text()) if compute_file.is_file() else None,
    'this_session': {'wall_time_h': round(hours, 3), 'cu_per_hour': CU_PER_HOUR,
                     'compute_units': round(hours * CU_PER_HOUR, 2)},
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
})
out = cp3.write_report(REPORT, paths.runs / 'checkpoints' / f'cp3_training_health_{PROFILE}_seed{SEED}.json')
for name, c in REPORT.get('checks', {}).items():
    print(f'{name:4} {c["status"]:13}', {k: v for k, v in c.items() if k != 'status'})
print('script verdict:', REPORT['verdict'], '| best epoch', REPORT.get('best_epoch'),
      '| run CU', (REPORT['compute_run'] or {}).get('compute_units'), '| saved', out)